In [1]:
# Cell 1
# Mục đích: tạo kết nối BigQuery và các hàm dùng chung (mb, stats) cho notebook mới.
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
# client là kênh kết nối; location phải khớp vùng dataset (US)
client = bigquery.Client(project=PROJECT, location="US")

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def stats(sql, max_mb=100):
    # chạy thật, tắt cache để số đo là số thật, đặt trần byte billed (cầu chì)
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,  # đổi MB sang byte
    )
    job = client.query(sql, job_config=cfg)   # gửi job, không chờ
    df = job.result().to_dataframe()          # chờ xong rồi đưa kết quả vào df
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df                                  # thiếu dòng này thì df = None

print("Cell 1 OK")

Cell 1 OK


In [2]:
# Cell 2
# Mục đích: đối chiếu biểu đồ cột Top 10 danh mục (năm 2026) và bảng biên trong Power BI.
sql = f"""
SELECT
  p.category,
  ROUND(SUM(f.sale_price), 2)  AS net_revenue,
  ROUND(SUM(f.gross_profit), 2) AS gross_profit,
  ROUND(SAFE_DIVIDE(SUM(f.gross_profit), SUM(f.sale_price)), 4) AS margin
FROM `{PROJECT}.dbt_dev.fct_net_sales` AS f
JOIN `{PROJECT}.dbt_dev.dim_products` AS p
  ON f.product_id = p.product_id
WHERE f.order_date BETWEEN '2026-01-01' AND '2026-12-31'
GROUP BY p.category
ORDER BY net_revenue DESC
LIMIT 10
"""
df = stats(sql)
print(len(df))   # mong đợi 10
df

processed: 1.95 MB | billed: 20.0 MB
10


,category,net_revenue,gross_profit,margin
0,Outerwear & Coats,118251.64,65524.61,0.5541
1,Jeans,108338.28,50278.14,0.4641
2,Sweaters,79970.26,41359.62,0.5172
3,Suits & Sport Coats,62583.55,37286.91,0.5958
4,Swim,58499.00,29034.96,0.4963
5,Fashion Hoodies & Sweatshirts,58252.66,27972.05,0.4802
6,Sleep & Lounge,49638.50,26004.98,0.5239
7,Shorts,47769.29,23831.03,0.4989
8,Dresses,44243.91,24330.84,0.5499
9,Tops & Tees,43609.76,19202.42,0.4403


In [3]:
# Cell 3
# Mục đích: đối chiếu hàng KPI năm 2026 và kiểm tra mốc dữ liệu cuối.
sql = f"""
SELECT
  ROUND(SUM(sale_price), 2)   AS net_revenue,
  ROUND(SUM(gross_profit), 2) AS gross_profit,
  ROUND(SAFE_DIVIDE(SUM(gross_profit), SUM(sale_price)), 4) AS margin,
  COUNT(DISTINCT order_id)    AS orders,
  ROUND(SAFE_DIVIDE(SUM(sale_price), COUNT(DISTINCT order_id)), 2) AS avg_order_value,
  MAX(order_date)             AS data_through,
  COUNT(DISTINCT order_date)  AS so_ngay_co_du_lieu
FROM `{PROJECT}.dbt_dev.fct_net_sales`
WHERE order_date BETWEEN '2026-01-01' AND '2026-12-31'
"""
df = stats(sql)
df.T   # xoay ngang để dễ đọc từng chỉ số

processed: 1.38 MB | billed: 10.0 MB


,0
net_revenue,982813.17
gross_profit,510954.59
margin,0.5199
orders,11083
avg_order_value,88.68
data_through,2026-10-22
so_ngay_co_du_lieu,295
